# Advanced Python for Battery Engineers with PyBaMM

> **Environment note:** these notebooks need a Python environment with a working
> PyBaMM install (`pip install pybamm numpy pandas scipy matplotlib ipykernel`).
> All simulations below use `pybamm.lithium_ion` models with the built-in
> `"Chen2020"` parameter set and run in well under a second each, so nothing
> here should take more than a few seconds to execute.

In [ ]:
%matplotlib inline
import time
import itertools
from functools import partial, lru_cache
from contextlib import contextmanager
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

import pybamm

pybamm.set_logging_level("WARNING")

## 1. Object-Oriented Battery Models

PyBaMM itself is built around class hierarchies: every model (`pybamm.lithium_ion.SPM`,
`pybamm.lithium_ion.DFN`, `pybamm.equivalent_circuit.Thevenin`, ...) is a subclass of
`pybamm.BaseModel`, and equivalent circuit models are literally *composed* of smaller
circuit-element submodels. This section builds the same kind of design — validated
properties, inheritance/polymorphism, and dataclasses — on small battery-engineering
examples, then connects a dataclass directly to a real `pybamm.ParameterValues` update.

### Example 1: A `Cell` class with validated properties

A plain attribute lets you assign `cell.capacity_ah = -5`, which is physically
meaningless. Wrapping the attribute in a `@property` with a setter lets us validate on
assignment while keeping the simple `cell.capacity_ah` syntax. `energy_wh` is a
*read-only* computed property.

In [ ]:
class Cell:
    """A minimal electrical rating for a battery cell."""

    def __init__(self, capacity_ah, nominal_voltage):
        self.capacity_ah = capacity_ah
        self.nominal_voltage = nominal_voltage

    @property
    def capacity_ah(self):
        return self._capacity_ah

    @capacity_ah.setter
    def capacity_ah(self, value):
        if value <= 0:
            raise ValueError("capacity_ah must be positive")
        self._capacity_ah = value

    @property
    def energy_wh(self):
        return self.capacity_ah * self.nominal_voltage

    def __repr__(self):
        return f"Cell({self.capacity_ah} Ah, {self.nominal_voltage} V)"


cell = Cell(5.0, 3.7)
print(cell, "->", cell.energy_wh, "Wh")

try:
    Cell(-1, 3.7)
except ValueError as e:
    print("Caught:", e)

### Example 2: Inheritance and polymorphism — composing an ECM

`pybamm.equivalent_circuit.Thevenin` is an R0 in series with one or more RC branches.
We can mirror that composition in plain Python: a common `CircuitComponent` interface
with a `voltage_drop` method, two concrete implementations (`Resistor`, `RCPair`), and
a container that treats every component polymorphically — it never needs to know which
concrete class it is calling.

In [ ]:
class CircuitComponent:
    """Common interface every equivalent-circuit element must implement."""

    def voltage_drop(self, current, dt=None):
        raise NotImplementedError


class Resistor(CircuitComponent):
    def __init__(self, resistance_ohm):
        self.resistance_ohm = resistance_ohm

    def voltage_drop(self, current, dt=None):
        return current * self.resistance_ohm


class RCPair(CircuitComponent):
    """One RC branch of a Thevenin-style equivalent circuit."""

    def __init__(self, resistance_ohm, capacitance_f):
        self.resistance_ohm = resistance_ohm
        self.capacitance_f = capacitance_f
        self.polarisation_v = 0.0

    def voltage_drop(self, current, dt):
        tau = self.resistance_ohm * self.capacitance_f
        self.polarisation_v += dt / tau * (current * self.resistance_ohm - self.polarisation_v)
        return self.polarisation_v


class TheveninECM:
    """Series stack of polymorphic circuit components, like pybamm's Thevenin model."""

    def __init__(self, components):
        self.components = components

    def terminal_voltage(self, ocv, current, dt):
        v = ocv
        for component in self.components:
            v -= component.voltage_drop(current, dt)
        return v


ecm = TheveninECM([Resistor(0.01), RCPair(0.02, 2000)])
v = ecm.terminal_voltage(ocv=3.9, current=5.0, dt=1.0)
print(f"Terminal voltage after 1 s at 5 A: {v:.4f} V")

### Example 3: A `CellDesign` dataclass feeding real PyBaMM parameters

`@dataclass` removes the boilerplate `__init__`/`__repr__` for simple data containers.
Here `CellDesign` stores two geometric design variables and knows how to translate
itself into the exact parameter names PyBaMM expects — the dataclass becomes the
bridge between "engineering variables I want to vary" and "PyBaMM's parameter
dictionary".

In [ ]:
@dataclass
class CellDesign:
    name: str
    negative_electrode_thickness_m: float
    positive_electrode_thickness_m: float

    def to_parameter_updates(self):
        return {
            "Negative electrode thickness [m]": self.negative_electrode_thickness_m,
            "Positive electrode thickness [m]": self.positive_electrode_thickness_m,
        }


baseline = CellDesign("baseline", 8.52e-05, 7.56e-05)
thicker_positive = CellDesign("thicker_positive", 8.52e-05, 7.56e-05 * 1.3)

for design in (baseline, thicker_positive):
    params = pybamm.ParameterValues("Chen2020")
    params.update(design.to_parameter_updates())
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    sol = sim.solve([0, 3600])
    print(f"{design.name}: V_end after 1 h = {sol['Terminal voltage [V]'].data[-1]:.4f} V")

## 2. Functional Python for Parameter Functions

Many PyBaMM parameters (current, diffusivity, OCV, exchange-current density, ...) are
plain Python *functions*, not just numbers — PyBaMM traces them symbolically to build
the model. That makes functional-programming tools (functions as first-class objects,
closures, `functools.partial`) directly useful: they let you *generate families* of
parameter functions instead of writing near-duplicate functions by hand.

One gotcha worth internalising early: inside a parameter function the arguments
(`t`, `sto`, `T`, ...) are symbolic PyBaMM expressions while the model is being built,
so use `pybamm.exp`, `pybamm.sin`, etc. instead of their NumPy equivalents.

### Example 1: Functions as first-class objects — a profile registry

A function is just another object: it can be stored in a dict, passed around, and
selected by name at runtime. Here a small registry of current-profile functions lets
`run_with_profile` stay generic — it never hard-codes which profile it runs.

In [ ]:
def step_current(t, magnitude=5.0):
    return magnitude + 0.0 * t


def ramp_current(t, start=1.0, rate_a_per_hour=0.5):
    return start + rate_a_per_hour * t / 3600


CURRENT_PROFILES = {
    "step": step_current,
    "ramp": ramp_current,
}


def run_with_profile(profile_name, **kwargs):
    profile_fn = CURRENT_PROFILES[profile_name]
    params = pybamm.ParameterValues("Chen2020")
    params["Current function [A]"] = lambda t: profile_fn(t, **kwargs)
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    return sim.solve([0, 1800])


sol_step = run_with_profile("step", magnitude=4.0)
sol_ramp = run_with_profile("ramp", start=1.0, rate_a_per_hour=2.0)
print("step  : V_end =", sol_step["Terminal voltage [V]"].data[-1])
print("ramp  : V_end =", sol_ramp["Terminal voltage [V]"].data[-1])

### Example 2: Closures — an Arrhenius diffusivity factory

`make_arrhenius_diffusivity` is a *factory function*: it takes engineering parameters
(`D_ref`, `Ea`) and returns a new function `D(sto, T)` that "remembers" them via a
closure. This lets us build several temperature-dependent diffusivity functions from
one template without repeating the Arrhenius formula each time.

In [ ]:
def make_arrhenius_diffusivity(D_ref, activation_energy_j_per_mol, T_ref=298.15):
    """Return a pybamm-compatible D(sto, T) closure following the Arrhenius law."""
    gas_constant = 8.314

    def D(sto, T):
        return D_ref * pybamm.exp(
            -activation_energy_j_per_mol / gas_constant * (1 / T - 1 / T_ref)
        )

    return D


D_low_Ea = make_arrhenius_diffusivity(D_ref=3.3e-14, activation_energy_j_per_mol=20000)
D_high_Ea = make_arrhenius_diffusivity(D_ref=3.3e-14, activation_energy_j_per_mol=50000)

for label, D_fn in [("low activation energy", D_low_Ea), ("high activation energy", D_high_Ea)]:
    params = pybamm.ParameterValues("Chen2020")
    params["Negative electrode diffusivity [m2.s-1]"] = D_fn
    params["Ambient temperature [K]"] = 278.15  # a cold start
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    sol = sim.solve([0, 1800])
    print(f"{label}: V_end at cold start = {sol['Terminal voltage [V]'].data[-1]:.4f} V")

### Example 3: `functools.partial` — freezing arguments to build variants

`functools.partial` takes a function and "freezes" some of its arguments, returning a
new callable. It's a lighter-weight alternative to writing a closure by hand when you
just need to fix a few keyword arguments — handy for turning one generic profile
function into several named variants.

In [ ]:
def sinusoidal_current(t, magnitude, period_s):
    return magnitude * (1 + pybamm.sin(2 * np.pi * t / period_s))


aggressive_profile = partial(sinusoidal_current, magnitude=6.0, period_s=200)
gentle_profile = partial(sinusoidal_current, magnitude=3.0, period_s=900)


def run_profile(current_fn):
    params = pybamm.ParameterValues("Chen2020")
    params["Current function [A]"] = current_fn
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    return sim.solve([0, 1800])


for label, profile in [("aggressive", aggressive_profile), ("gentle", gentle_profile)]:
    sol = run_profile(profile)
    print(f"{label}: capacity discharged = {sol['Discharge capacity [A.h]'].data[-1]:.4f} Ah")

## 3. Iterators, Generators, Decorators and Context Managers

Battery engineering work is full of *sequences to build* (cycling protocols, sweep
grids) and *blocks to wrap* (time this, log that, cache this expensive call). Python's
generators, `itertools`, decorators and context managers are the idiomatic tools for
exactly this, and they compose cleanly with PyBaMM's `Simulation`/`Experiment` API.

### Example 1: A generator for a cycling protocol

A generator function *yields* steps lazily instead of building a full list up front.
For `pybamm.Experiment` this doesn't matter much at small scale, but the pattern scales
to protocols with thousands of steps without ever holding them all in memory at once.

In [ ]:
def cccv_cycle_steps(n_cycles, charge_c_rate=0.5, discharge_c_rate=1.0):
    for _ in range(n_cycles):
        yield f"Charge at {charge_c_rate}C until 4.2 V"
        yield f"Hold at 4.2 V until C/50"
        yield f"Discharge at {discharge_c_rate}C until 3.0 V"


steps = list(cccv_cycle_steps(2))
experiment = pybamm.Experiment(steps)
print(f"Generated {len(steps)} steps for 2 CCCV cycles")
print("First step:", steps[0])

### Example 2: `itertools.product` for sweep grids

Nested `for` loops for a parameter grid get unwieldy past two dimensions.
`itertools.product` flattens any number of dimensions into one clean loop, and — being
an iterator — never materialises the whole grid in memory.

In [ ]:
c_rates = [0.5, 1.0, 2.0]
temperatures_k = [278.15, 298.15]

results = []
for c_rate, temperature_k in itertools.product(c_rates, temperatures_k):
    params = pybamm.ParameterValues("Chen2020")
    nominal_capacity = params["Nominal cell capacity [A.h]"]
    params["Current function [A]"] = nominal_capacity * c_rate
    params["Ambient temperature [K]"] = temperature_k
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    sol = sim.solve([0, 600])
    results.append(
        {"c_rate": c_rate, "temperature_K": temperature_k, "v_end": sol["Terminal voltage [V]"].data[-1]}
    )

for r in results:
    print(f"C-rate={r['c_rate']:.1f}, T={r['temperature_K']:.2f} K -> V_end={r['v_end']:.4f} V")

### Example 3: A context manager for timing simulations

`@contextmanager` turns a generator function into a `with`-block: code before `yield`
runs on entry, code after `yield` runs on exit (even if an exception occurs). This is
the idiomatic way to wrap "do something, then always clean up / report" logic.

In [ ]:
@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label}: {elapsed:.4f} s")


with timer("SPM solve"):
    pybamm.Simulation(pybamm.lithium_ion.SPM()).solve([0, 3600])

with timer("DFN solve"):
    pybamm.Simulation(pybamm.lithium_ion.DFN()).solve([0, 3600])

### Example 4: Decorators for timing and memoized caching

A decorator wraps a function to add behaviour without changing its body. `@timeit`
below is a general-purpose timing decorator; `@memoize` caches results in a dict keyed
by arguments, so re-running an expensive simulation for a C-rate you've already solved
is instant the second time.

In [ ]:
def timeit(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__}{args}: {time.perf_counter() - start:.4f} s")
        return result

    return wrapper


def memoize(func):
    cache = {}

    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]

    wrapper.cache = cache
    return wrapper


@timeit
@memoize
def capacity_at_c_rate(c_rate):
    params = pybamm.ParameterValues("Chen2020")
    nominal_capacity = params["Nominal cell capacity [A.h]"]
    params["Current function [A]"] = nominal_capacity * c_rate
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    sol = sim.solve([0, 3600 / c_rate])
    return sol["Discharge capacity [A.h]"].data[-1]


capacity_at_c_rate(1.0)  # first call: actually solves
capacity_at_c_rate(1.0)  # second call: served from the cache, effectively instant

## 4. NumPy Vectorization for Post-Processing

A `pybamm.Solution` hands you plain NumPy arrays via `solution["Variable name"].data`
(and `.entries` for higher-dimensional/spatial variables). From there, everyday battery
post-processing — capacity integration, incremental capacity analysis, comparing two
voltage curves — is just vectorized NumPy, no Python-level loops required.

### Example 1: Discharge capacity by integrating current

`np.trapezoid` (the NumPy 2.x replacement for the now-removed `np.trapz`) performs
trapezoidal integration over an array in one vectorized call — the same computation
PyBaMM itself does internally to track `"Discharge capacity [A.h]"`.

In [ ]:
sim = pybamm.Simulation(pybamm.lithium_ion.SPM())
sol = sim.solve([0, 3600])

t = sol["Time [s]"].data
current = sol["Current [A]"].data
capacity_ah = np.trapezoid(current, t) / 3600

print(f"Integrated manually : {capacity_ah:.4f} Ah")
print(f"PyBaMM's own tracker: {sol['Discharge capacity [A.h]'].data[-1]:.4f} Ah")

### Example 2: Incremental capacity analysis (dQ/dV)

`np.gradient` computes a vectorized numerical derivative in one call (central
differences in the interior, one-sided at the edges) — much less error-prone than a
hand-rolled finite-difference loop, and it plots straight into a standard dQ/dV curve.

In [ ]:
voltage = sol["Terminal voltage [V]"].data
capacity = sol["Discharge capacity [A.h]"].data

with np.errstate(divide="ignore", invalid="ignore"):
    dqdv = np.gradient(capacity, voltage)

plt.figure()
plt.plot(voltage, dqdv)
plt.xlabel("Voltage [V]")
plt.ylabel("dQ/dV [Ah/V]")
plt.title("Incremental capacity analysis")
plt.show()

### Example 3: Vectorized RMSE between two voltage curves

Two solvers don't report at the same time points, so a direct element-wise comparison
isn't possible. `np.interp` resamples both curves onto a common time grid, after which
the RMSE is a single vectorized expression.

In [ ]:
sol_spm = pybamm.Simulation(pybamm.lithium_ion.SPM()).solve([0, 3600])
sol_dfn = pybamm.Simulation(pybamm.lithium_ion.DFN()).solve([0, 3600])

t_common = np.linspace(0, 3600, 200)
v_spm = np.interp(t_common, sol_spm["Time [s]"].data, sol_spm["Terminal voltage [V]"].data)
v_dfn = np.interp(t_common, sol_dfn["Time [s]"].data, sol_dfn["Terminal voltage [V]"].data)

rmse_v = np.sqrt(np.mean((v_spm - v_dfn) ** 2))
print(f"RMSE between SPM and DFN voltage: {rmse_v * 1000:.2f} mV")

## 5. Pandas for Parameter Sweeps and Tidy Data

A single simulation gives you one `Solution`; a *sweep* gives you many, and the
natural container for "many rows of results, each with a few labelled columns" is a
tidy `pandas.DataFrame`. This section builds sweep results directly into DataFrames,
then uses `groupby`/`pivot`/`merge` to summarise and compare them — the same toolkit
you'd use on real test-bench data.

### Example 1: Sweeping C-rates into a tidy DataFrame

Collecting each simulation's results as a dict, then passing the list of dicts to
`pd.DataFrame`, is the simplest reliable way to build a tidy sweep table.

In [ ]:
def sweep_c_rates(c_rates):
    rows = []
    for c_rate in c_rates:
        params = pybamm.ParameterValues("Chen2020")
        nominal_capacity = params["Nominal cell capacity [A.h]"]
        params["Current function [A]"] = nominal_capacity * c_rate
        sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
        sol = sim.solve([0, 3600 / c_rate])
        rows.append(
            {
                "c_rate": c_rate,
                "v_end": sol["Terminal voltage [V]"].data[-1],
                "capacity_ah": sol["Discharge capacity [A.h]"].data[-1],
            }
        )
    return pd.DataFrame(rows)


sweep_df = sweep_c_rates([0.2, 0.5, 1.0, 2.0, 3.0])
sweep_df

### Example 2: `groupby` and `pivot` to summarise a 2D sweep

For a two-factor sweep (C-rate x temperature), `pivot` reshapes the tidy (long-format)
table into a wide matrix that reads like a lookup table — exactly the shape you'd want
for a capacity-vs-C-rate-vs-temperature datasheet table.

In [ ]:
rows = []
for c_rate, temperature_k in itertools.product([0.5, 1.0, 2.0], [278.15, 298.15, 313.15]):
    params = pybamm.ParameterValues("Chen2020")
    nominal_capacity = params["Nominal cell capacity [A.h]"]
    params["Current function [A]"] = nominal_capacity * c_rate
    params["Ambient temperature [K]"] = temperature_k
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    sol = sim.solve([0, 3600 / c_rate])
    rows.append(
        {
            "c_rate": c_rate,
            "temperature_K": temperature_k,
            "capacity_ah": sol["Discharge capacity [A.h]"].data[-1],
        }
    )

grid_df = pd.DataFrame(rows)
capacity_table = grid_df.pivot(index="c_rate", columns="temperature_K", values="capacity_ah")
capacity_table

### Example 3: Comparing simulation results against reference data

Real workflows usually need to compare a simulation sweep against bench-test or
datasheet numbers. `DataFrame.merge` aligns the two on a shared key, after which the
percentage error is just vectorized column arithmetic.

In [ ]:
reference_df = pd.DataFrame(
    {"c_rate": [0.2, 0.5, 1.0, 2.0, 3.0], "capacity_ah": [5.02, 4.99, 4.93, 4.78, 4.55]}
)

comparison_df = sweep_df.merge(reference_df, on="c_rate", suffixes=("_sim", "_ref"))
comparison_df["pct_error"] = (
    100 * (comparison_df["capacity_ah_sim"] - comparison_df["capacity_ah_ref"]) / comparison_df["capacity_ah_ref"]
)
comparison_df.sort_values("pct_error", ascending=False)

## 6. Optimization and Design-Space Exploration (Capstone)

This capstone section combines everything above — a dataclass-like design
representation, a caching decorator, vectorized post-processing and a tidy DataFrame —
around a single realistic question: *what electrode thickness maximises energy density
without dropping capacity too far?* We solve it first by brute-force grid search, then
with `scipy.optimize`, then wrap the whole thing in one reusable class.

### Example 1: Grid search over positive electrode thickness

The simplest optimization strategy: evaluate the objective on a grid of candidate
values and pick the best. It's exhaustive rather than efficient, but it's easy to
reason about and — like Section 5 — produces a DataFrame you can inspect and plot.

In [ ]:
def energy_density_wh_per_kg(thickness_m, mass_scale_kg_per_m=5000):
    params = pybamm.ParameterValues("Chen2020")
    params["Positive electrode thickness [m]"] = thickness_m
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    sol = sim.solve([0, 3600])
    t = sol["Time [s]"].data
    power_w = sol["Terminal voltage [V]"].data * sol["Current [A]"].data
    energy_wh = np.trapezoid(power_w, t) / 3600
    mass_kg = thickness_m * mass_scale_kg_per_m
    return energy_wh / mass_kg


thicknesses = np.linspace(40e-6, 100e-6, 13)
grid_results = pd.DataFrame(
    {"thickness_um": thicknesses * 1e6, "energy_density": [energy_density_wh_per_kg(x) for x in thicknesses]}
)
best_row = grid_results.loc[grid_results["energy_density"].idxmax()]

plt.figure()
plt.plot(grid_results["thickness_um"], grid_results["energy_density"], "o-")
plt.axvline(best_row["thickness_um"], color="red", linestyle="--", label="grid best")
plt.xlabel("Positive electrode thickness [um]")
plt.ylabel("Energy density [Wh/kg] (proxy)")
plt.legend()
plt.show()

print(f"Grid-search best: {best_row['thickness_um']:.1f} um -> {best_row['energy_density']:.3f} Wh/kg")

### Example 2: Refining with `scipy.optimize.minimize_scalar`

Grid search only ever finds a point *on the grid*. `minimize_scalar` with
`method="bounded"` uses golden-section search to converge on the true optimum within
a bounded interval — usually in far fewer model evaluations than a fine grid would
need. `scipy.optimize` only minimizes, so maximizing `energy_density_wh_per_kg` means
minimizing its negative.

In [ ]:
result = minimize_scalar(
    lambda x: -energy_density_wh_per_kg(x),
    bounds=(40e-6, 100e-6),
    method="bounded",
    options={"xatol": 1e-7},
)

print(f"scipy-optimized best: {result.x * 1e6:.3f} um -> {-result.fun:.3f} Wh/kg")
print(f"Number of model evaluations: {result.nfev}")

### Example 3: Putting it together — a `CellDesignOptimizer` class

The final step wraps the objective, a capacity *constraint*, and a cache into one
class — an OOP capstone that reuses the memoization pattern from Section 3 and the
optimization pattern from Example 2 above, applied to a design decision an engineer
actually cares about.

In [ ]:
class CellDesignOptimizer:
    """Finds the positive electrode thickness maximising energy density subject to
    a minimum discharge capacity."""

    def __init__(self, min_capacity_ah, mass_scale_kg_per_m=5000):
        self.min_capacity_ah = min_capacity_ah
        self.mass_scale_kg_per_m = mass_scale_kg_per_m
        self._cache = {}

    def _simulate(self, thickness_m):
        if thickness_m not in self._cache:
            params = pybamm.ParameterValues("Chen2020")
            params["Positive electrode thickness [m]"] = thickness_m
            sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
            self._cache[thickness_m] = sim.solve([0, 3600])
        return self._cache[thickness_m]

    def objective(self, thickness_m):
        sol = self._simulate(thickness_m)
        t = sol["Time [s]"].data
        power_w = sol["Terminal voltage [V]"].data * sol["Current [A]"].data
        energy_wh = np.trapezoid(power_w, t) / 3600
        mass_kg = thickness_m * self.mass_scale_kg_per_m
        energy_density = energy_wh / mass_kg

        capacity_ah = sol["Discharge capacity [A.h]"].data[-1]
        if capacity_ah < self.min_capacity_ah:
            energy_density -= 1000 * (self.min_capacity_ah - capacity_ah)
        return energy_density

    def optimize(self, bounds=(40e-6, 100e-6)):
        result = minimize_scalar(lambda x: -self.objective(x), bounds=bounds, method="bounded")
        return result.x


optimizer = CellDesignOptimizer(min_capacity_ah=4.9)
best_thickness = optimizer.optimize()
print(f"Constrained optimum: {best_thickness * 1e6:.3f} um")
print(f"Unique simulations run (thanks to caching): {len(optimizer._cache)}")

## Conclusion

This notebook walked through six areas of "advanced" Python, each grounded in a
battery-engineering task and, from Section 2 onward, real PyBaMM simulations:

1. **Object-oriented design** — validated properties, inheritance/polymorphism for
   composable circuit models, and `@dataclass` as a bridge to `pybamm.ParameterValues`.
2. **Functional programming** — functions as first-class objects, closures for
   parameterised PyBaMM parameter functions, and `functools.partial` for building
   families of profile functions.
3. **Iterators, generators, decorators and context managers** — lazily generating
   cycling protocols and sweep grids, timing/caching simulations with decorators, and
   `@contextmanager` for clean "wrap this block" logic.
4. **NumPy vectorization** — `np.trapezoid` for capacity integration, `np.gradient`
   for incremental capacity analysis, and `np.interp` for comparing curves on a common
   grid.
5. **pandas for tidy data** — building sweep results directly into DataFrames, then
   `groupby`/`pivot`/`merge` to summarise and compare against reference data.
6. **Optimization** — grid search vs. `scipy.optimize.minimize_scalar`, and a capstone
   `CellDesignOptimizer` class combining a cache, an objective and a constraint into a
   reusable design-space exploration tool.

The companion **Exercises** notebook has 3 hands-on problems per section (18 total) —
each test cell already shows the target output from a working reference solution, so
you can check your implementation as you go. The **Solutions** notebook has every
problem fully worked and executed.

### Further reading
- [PyBaMM documentation home](https://docs.pybamm.org/)
- [PyBaMM: Simulation class reference](https://docs.pybamm.org/en/latest/source/api/simulation.html)
- [PyBaMM: ParameterValues reference](https://docs.pybamm.org/en/latest/source/api/parameters/parameter_values.html)
- [PyBaMM: Experiment reference](https://docs.pybamm.org/en/latest/source/api/experiment/experiment.html)
- [PyBaMM example notebooks index](https://docs.pybamm.org/en/latest/source/examples/index.html)
- [Python: dataclasses](https://docs.python.org/3/library/dataclasses.html)
- [Python: itertools](https://docs.python.org/3/library/itertools.html)
- [Python: functools](https://docs.python.org/3/library/functools.html)
- [Python: contextlib](https://docs.python.org/3/library/contextlib.html)
- [NumPy user guide](https://numpy.org/doc/stable/user/index.html)
- [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html)
- [SciPy optimize reference](https://docs.scipy.org/doc/scipy/reference/optimize.html)